# 第10章 性能評価

## 10.2 評価指標を用いた自動評価

### 10.2.4 多肢選択式質問応答タスクによる自動評価

#### 環境準備

In [2]:
!pip install transformers[torch,sentencepiece] bitsandbytes 'datasets<4.0.0'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 47.5 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [3]:
from transformers.trainer_utils import set_seed
set_seed(42)

#### データセットの準備

In [5]:
from datasets import load_dataset
train_dataset = load_dataset(
    "llm-book/JGLUE", name="JCommonsenseQA", split="train"
)
val_dataset = load_dataset(
    "llm-book/JGLUE", name="JCommonsenseQA", split="validation"
)
print(val_dataset)

README.md:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

JGLUE.py:   0%|          | 0.00/13.9k [00:00<?, ?B/s]

preprocess_marc_ja.py:   0%|          | 0.00/9.03k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['q_id', 'question', 'choice0', 'choice1', 'choice2', 'choice3', 'choice4', 'label'],
    num_rows: 1119
})


In [6]:
print(train_dataset[0])

{'q_id': 0, 'question': '主に子ども向けのもので、イラストのついた物語が書かれているものはどれ？', 'choice0': '世界', 'choice1': '写真集', 'choice2': '絵本', 'choice3': '論文', 'choice4': '図鑑', 'label': 2}


#### データの前処理

In [7]:
from pprint import pprint

def convert_data_format(data: dict[str, str]) -> dict[str, str]:
    """選択肢の中から質問に数字で回答する形式にデータを変換する"""
    data["input"] = (
        f"質問：{data['question']}\n"
        f"選択肢：0.{data['choice0']},1.{data['choice1']},"
        f"2.{data['choice2']},3.{data['choice3']},"
        f"4.{data['choice4']}"
    )
    data["output"] = data["label"]
    return data

In [8]:
# 訓練セットをシャッフルする
train_dataset = train_dataset.shuffle()
# 訓練セットの前処理をする
train_dataset = train_dataset.map(convert_data_format)
# 4つのfew-shot事例を取得する
few_shots = list(train_dataset)[:4]
# 検証セットの前処理をする
val_dataset = val_dataset.map(convert_data_format)
pprint(list(val_dataset)[0])

Map:   0%|          | 0/8939 [00:00<?, ? examples/s]

Map:   0%|          | 0/1119 [00:00<?, ? examples/s]

{'choice0': '掲示板',
 'choice1': 'パソコン',
 'choice2': 'マザーボード',
 'choice3': 'ハードディスク',
 'choice4': 'まな板',
 'input': '質問：電子機器で使用される最も主要な電子回路基板の事をなんと言う？\n'
          '選択肢：0.掲示板,1.パソコン,2.マザーボード,3.ハードディスク,4.まな板',
 'label': 2,
 'output': 2,
 'q_id': 8939,
 'question': '電子機器で使用される最も主要な電子回路基板の事をなんと言う？'}


In [9]:
print(few_shots[0])

{'q_id': 8530, 'question': '4輪でハンドルで操作し、ガソリンや電気で動く乗り物は？', 'choice0': '自動車', 'choice1': '自転車', 'choice2': 'イヤホン', 'choice3': '飛行機', 'choice4': 'ライブ', 'label': 0, 'input': '質問：4輪でハンドルで操作し、ガソリンや電気で動く乗り物は？\n選択肢：0.自動車,1.自転車,2.イヤホン,3.飛行機,4.ライブ', 'output': 0}


#### プロンプトテンプレートの作成

In [10]:
def create_prompt_template(
    instruction: str,
    few_shots: list[dict[str, str]] | None = None
) -> str:
    """プロンプトテンプレートを作成する"""
    prompt_template = (
        "以下は、タスクを説明する指示と、"
        "文脈のある入力の組み合わせです。"
        "要求を適切に満たす応答を書きなさい。\n\n"
    )
    prompt_template += f"### 指示\n{instruction}\n\n"
    if few_shots is not None:
        for few_shot in few_shots:
            prompt_template += f"### 入力:\n{few_shot["input"]}\n\n"
            prompt_template += f"### 応答:\n{few_shot["output"]}\n\n"
    prompt_template += "### 入力:\n{input}\n\n"
    prompt_template += "### 応答:\n"
    return prompt_template

In [11]:
# 指示文を指定してプロンプトテンプレートを作成する
instruction = """
質問と解答の選択肢を入力として受け取り、選択肢から回答を選択してください。
なお、回答は選択肢の番号（例：0）でするものとします。 
回答となる数値をint型で返し、他には何も含めないことを厳守してください。
"""
prompt_template = create_prompt_template(instruction, few_shots)
print(prompt_template)

以下は、タスクを説明する指示と、文脈のある入力の組み合わせです。要求を適切に満たす応答を書きなさい。

### 指示

質問と解答の選択肢を入力として受け取り、選択肢から回答を選択してください。
なお、回答は選択肢の番号（例：0）でするものとします。 
回答となる数値をint型で返し、他には何も含めないことを厳守してください。


### 入力:
質問：4輪でハンドルで操作し、ガソリンや電気で動く乗り物は？
選択肢：0.自動車,1.自転車,2.イヤホン,3.飛行機,4.ライブ

### 応答:
0

### 入力:
質問：夏になったら着たくなるものは？
選択肢：0.誘い水,1.水着,2.化粧水,3.水すまし,4.水筒

### 応答:
1

### 入力:
質問：声や楽器を使った芸術は？
選択肢：0.猫,1.音楽,2.展覧会,3.歌,4.スズメ

### 応答:
1

### 入力:
質問：売る物のことを何と言うか？
選択肢：0.自転車,1.鍋,2.車,3.飲み物,4.商品

### 応答:
4

### 入力:
{input}

### 応答:



#### パイプラインの作成